# When the arrow points both ways

Price moves quantity and quantity moves price. Ask what the demand slope is and every
ordinary regression you can write down gives an answer that is not it — not noisy, *biased*,
in a known direction, by an amount that does not shrink with more data.

A causal DAG is a *solved* model: it assumes you can order the variables so every arrow
points forward. Two ordinary situations break that assumption without being ill-posed —
**simultaneity** (two quantities determined together) and **time structure** (yesterday's
value entering today's equation). `axiom.dynamics` takes such a system as a declarative
`Spec` and *compiles* it into ordinary `axiom.core.expr` trees, so there is still exactly one
`forward()` and the identification machinery downstream sees an acyclic graph.

In [ ]:
from axiom.core import D, Param, dimensionless, latex, value
from axiom.dynamics import (
    AffineForm, Block, BlockOrder, BlockSolution, DynamicEquation, DynamicSystem, DynamicsError,
    Form, LeadNotSupportedError, ParseError, Role, SolveMethod, Unrolled, Variable, affine_split,
    block_order, conditional_form, lag_ref, lagged_columns, parse_equations, parse_ref,
    parse_system, prepare_panel, reduced_form, refs_in, solve_block, strongly_connected_components,
    time_ref, to_model_spec, unroll, unrolled_edges,
)
import numpy as np
import pandas as pd

from axiom.display import enable, show_math, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import CRITICAL, caption, compare, curve_band, intervals, mark_x

enable();  # every axiom result renders itself from here on

NONE = dimensionless()

## Declaring a system

Variables carry a dimension and a role: `endogenous` means an equation determines it,
`exogenous` means it is supplied. `observed=False` marks a latent state. `initial` is the
value lags take before the first period.

The equations get a small syntax — that is the part that is unreadable as a tree. Names
resolve to a parameter if one is declared and to a variable otherwise; `stock[t-1]` (or
`stock.l1`) is a lag.

In [ ]:
system = parse_system(
    "stock = decay * stock[t-1] + beta * inflow",
    variables=(
        Variable(name="stock", dimension=D.outcome, initial=0.0, description="the compartment"),
        Variable(name="inflow", dimension=D.currency, role="exogenous"),
    ),
    parameters=(
        Param(name="decay", dimension=NONE),
        Param(name="beta", dimension=D.outcome / D.currency),
    ),
    name="one-compartment",
)
print(system.endogenous, system.exogenous, "| max lag:", system.max_lag)
print([p.name for p in system.parameters])
show_math(system.equation("stock").rhs)

In [ ]:
# References parse both ways, and an equation knows what it reads.
print(lag_ref("stock", 2), parse_ref("stock.l2"), time_ref("stock", 3))
print(refs_in(system.equation("stock").rhs))
print(system.equation("stock").refs, "| label:", system.equation("stock").label)
role: Role = system.variable("inflow").role
print("role of inflow:", role)

## What the language refuses

Every refusal names what is wrong and what would fix it. Dimensions are checked across the
equals sign, using the same dimension interpreter the rest of axiom uses — a system that
type-checks here cannot disagree with `forward()` later.

In [ ]:
try:
    parse_system(
        "y = x",
        variables=(
            Variable(name="y", dimension=D.outcome),
            Variable(name="x", dimension=D.currency, role="exogenous"),
        ),
    )
except Exception as e:
    print(type(e).__name__, "->", e)

try:
    parse_ref("y.f1")
except LeadNotSupportedError as e:
    print("lead ->", e)

try:
    parse_equations("y = wobble", variables=(Variable(name="y", dimension=NONE),))
except ParseError as e:
    print("unknown name ->", e)

try:
    DynamicSystem(variables=(Variable(name="y", dimension=NONE),), equations=())
except Exception as e:
    print("missing equation ->", e)

## Blocks: which equations are genuinely simultaneous

The strongly connected components of the contemporaneous dependency graph are exactly the
sets that have to be solved together. A recursive system is already a DAG within a period.
Lagged edges never enter a block — which is precisely why unrolling gives a DAG.

In [ ]:
order: BlockOrder = block_order(system)
print("recursive:", order.recursive, "| blocks:", [(b.variables, b.simultaneous) for b in order.blocks])
first: Block = order.blocks[0]
print("first block size:", first.size, "| solution order:", order.order, "| largest:", order.largest_block)
print(strongly_connected_components(("a", "b", "c"), (("a", "b"), ("b", "a"), ("b", "c"))))

In [ ]:
market = parse_system(
    """
    quantity = a - b * price + c * income + q_shock
    price    = d + e * quantity + cost + p_shock
    """,
    variables=(
        Variable(name="quantity", dimension=D.outcome),
        Variable(name="price", dimension=D.currency),
        Variable(name="income", dimension=D.currency, role="exogenous"),
        Variable(name="cost", dimension=D.currency, role="exogenous"),
        Variable(name="q_shock", dimension=D.outcome, role="exogenous"),
        Variable(name="p_shock", dimension=D.currency, role="exogenous"),
    ),
    parameters=(
        Param(name="a", dimension=D.outcome),
        Param(name="b", dimension=D.outcome / D.currency),
        Param(name="c", dimension=D.outcome / D.currency),
        Param(name="d", dimension=D.currency),
        Param(name="e", dimension=D.currency / D.outcome),
    ),
    name="market",
)
market_order = block_order(market)
print("recursive:", market_order.recursive)
print("simultaneous:", [b.variables for b in market_order.simultaneous_blocks])
print("cycles through:", market.cycles_through())

## Solving a block

A simultaneous block is solved *symbolically at compile time*. `affine_split` decides,
structurally, whether each right-hand side is affine in the block's unknowns; if it is,
`reduced_form` eliminates them and the result is the econometric reduced form — exact for
every parameter value, and differentiable, so the design math downstream sees the true
derivative.

In [ ]:
from axiom.core import Data, Mul

form: AffineForm | None = affine_split(market.equation("quantity").rhs, ["quantity", "price"])
print("affine in the unknowns:", form is not None, "| coefficients on:", sorted(form.coefficients))

dims = {v.name: v.dimension for v in market.variables}
solved = reduced_form(
    {v: market.equation(v).rhs for v in ("quantity", "price")},
    dims,
)
show_math(solved["quantity"])

### What that solve is worth, in the only currency that matters

Below, data generated from this market: demand slope `b = 1.4`, supply feeding back through
`e`, both equations shocked. Then two estimates of the demand slope — the regression of
quantity on price that everybody's first pass produces, and the one the reduced form licenses.

In [ ]:
rng = np.random.default_rng(0)
truth = {"a": 40.0, "b": 1.4, "c": 0.5, "d": 5.0, "e": 0.35}
n = 400
world = {
    "income": rng.normal(20.0, 6.0, n),
    "cost": rng.normal(4.0, 1.5, n),
    "q_shock": rng.normal(0.0, 3.0, n),
    "p_shock": rng.normal(0.0, 1.2, n),
}
q = np.ravel(value(solved["quantity"], data=world, params=truth))
p = np.ravel(value(solved["price"], data=world, params=truth))

naive = -np.polyfit(p, q, 1)[0]                       # regress quantity on price
# the reduced form says d quantity / d income = c / (1 + b·e); invert it for b
dq_dincome = np.polyfit(world["income"], q, 1)[0]
recovered = (truth["c"] / dq_dincome - 1) / truth["e"]

fig = compare(
    ["regression of quantity on price", "via the reduced form", "truth"],
    [naive, recovered, truth["b"]],
    highlight="truth",
    value_fmt="{:.2f}",
    title="The demand slope, three ways",
    subtitle="400 periods of a market where price and quantity are determined together",
    x_title="estimated b (demand slope)",
)
caption(fig, "The first bar is not noisy — it is biased, and it stays biased as n grows, "
             "because price is not exogenous to quantity. Declaring the simultaneity is what "
             "makes the third bar reachable.")

In [ ]:
solution: BlockSolution = solve_block(
    {v: market.equation(v).rhs for v in ("quantity", "price")},
    dims,
    simultaneous=True,
)
method: SolveMethod = solution.method
print("method:", method, "| exact:", solution.exact, "| nodes:", solution.node_count)
print("residuals (empty because the solve is exact):", solution.residuals)

### A nonlinear block is approximate, and says so

When a right-hand side is not affine there is no closed-form reduced form. The block is
compiled as a declared number of Gauss-Seidel sweeps, `exact` is false, and `residuals`
carries `rhs(v) - v` per unknown — evaluate it on real data and you get the error you are
actually running. The tree grows *geometrically* in the sweeps, so useful counts are single
digits; past the node budget the answer is `Unsupported`, not a hang.

In [ ]:
saturating = parse_system(
    """
    response = k * load / (1 + load)
    load     = drive + g * response
    """,
    variables=(
        Variable(name="load", dimension=NONE),
        Variable(name="response", dimension=NONE),
        Variable(name="drive", dimension=NONE, role="exogenous"),
    ),
    parameters=(Param(name="k", dimension=NONE), Param(name="g", dimension=NONE)),
    name="saturating-feedback",
)
print("without sweeps:", conditional_form(saturating).reason[:120])

swept = conditional_form(saturating, sweeps=6)
print("with 6 sweeps -> exact:", swept.exact, "| nodes:", swept.node_count)
data, theta = {"drive": np.array([1.0])}, {"k": 2.0, "g": 0.5}
got = float(np.ravel(value(swept.expression("response"), data=data, params=theta))[0])
worst = max(
    abs(float(np.ravel(value(r, data=data, params=theta))[0]))
    for r in swept.approximate_blocks[0].residuals.values()
)
fixed_point = 0.0
for _ in range(400):
    fixed_point = 2.0 * (1 + 0.5 * fixed_point) / (1 + 1 + 0.5 * fixed_point)
print(f"response {got:.6f} vs fixed point {fixed_point:.6f}; largest residual {worst:.2e}")

In [ ]:
counts, errors, nodes = [], [], []
for sweeps in range(1, 9):
    compiled = conditional_form(saturating, sweeps=sweeps)
    val = float(np.ravel(value(compiled.expression("response"), data=data, params=theta))[0])
    counts.append(sweeps)
    errors.append(abs(val - fixed_point))
    nodes.append(compiled.node_count)

fig = curve_band(
    counts, np.log10(np.maximum(errors, 1e-16)),
    label="log₁₀ error",
    title="Approximate, and it tells you by how much",
    subtitle="distance from the true fixed point against Gauss-Seidel sweeps (tree size in the caption)",
    x_title="sweeps", y_title="log₁₀ |error|",
)
caption(fig, f"Node count over the same range: {nodes[0]} → {nodes[-1]}. The tree grows "
             f"geometrically while the error falls geometrically, which is why the sweep "
             f"count is a declared number and the residuals ship with the compiled system.")

## What this bought you

Feedback and lags declared once, in a syntax that reads like the model, checked for
dimensional sense, and compiled into the same expression trees everything else in axiom
evaluates. Exactly where a closed-form solve exists, you get one; where it does not, you get
a stated approximation with its residuals rather than a silent iteration limit.

`02-unrolling-and-fitting.ipynb` takes the compiled system to a panel and a posterior.